In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
import os
dega.__version__

'0.8.1'

## Download data if needed

Data source: https://www.10xgenomics.com/datasets/ffpe-human-pancreas-with-xenium-multimodal-cell-segmentation-1-standard

In [3]:
# ! curl -o ../data/Xenium_V1_human_Pancreas_FFPE_outs.zip https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_human_Pancreas_FFPE/Xenium_V1_human_Pancreas_FFPE_outs.zip
# ! unzip ../data/Xenium_V1_human_Pancreas_FFPE_outs.zip -d ../data/Xenium_V1_human_Pancreas_FFPE_outs

## Run preprocessing


To run the whole thing in command line
- Git clone celldega repo and 
- cd to celldega/src/celldega, then run:

python src/celldega/pre/run_pre_processing.py \
    --sample Xenium_V1_human_Pancreas_FFPE_outs \
    --data_root_dir data \
    --tile_size 250 \
    --image_tile_layer 'all' \
    --path_landscape_files notebooks/Xenium_V1_human_Pancreas_FFPE_outs


In [4]:
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'

# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

data_root_dir='../data'
tile_size=250
image_tile_layer='all'
path_landscape_files=f'landscape_files/{sample}'

dega.pre.main(
    sample=sample,
    data_root_dir=data_root_dir,
    tile_size=tile_size,
    image_tile_layer=image_tile_layer,
    path_landscape_files=path_landscape_files,
    )


Starting preprocessing for sample: Xenium_V1_human_Pancreas_FFPE_outs

========Unzip and extract Xenium-related files========
All files have been successfully extracted or skipped.

========Check if all required files or directories exist========
All required files or directories for technology 'Xenium' are present in '../data/Xenium_V1_human_Pancreas_FFPE_outs'.

========Save cbg gene parquet========
Processing gene 0: ABCC11
Processing gene 100: CLECL1
Processing gene 200: IL1RL1
Processing gene 300: RGS16
Processing gene 400: NegControlCodeword_0503
Processing gene 500: UnassignedCodeword_0459

========Write meta gene files========
cbg is a dense DataFrame. Proceeding with dense operations.
Calculating mean expression
Calculating variance
All meta gene files are succesfully saved.

========Write xenium transform file from the Zarr folder========
Transformation matrix saved to 'landscape_files/Xenium_V1_human_Pancreas_FFPE_outs/micron_to_image_transform.csv'.

========Make meta cells

Processing chunks: 100%|███████████████████████| 81/81 [00:00<00:00, 372.02it/s]
Processing coarse tiles: 84tile [00:24,  3.47tile/s]


tile bounds: {'x_min': 0, 'x_max': 34126.65, 'y_min': 0, 'y_max': 13744.4}

========Generating boundary tiles========


Processing coarse tiles: 100%|██████████████████| 14/14 [00:32<00:00,  2.31s/it]



========Create cluster gene expression (df_sig)========
Cluster-specific gene expression signatures saved successfully.

========Save landscape parameters========
Done.
Preprocessing completed successfully.


/Users/whuan/opt/anaconda3/envs/celldega_env_2025/lib/python3.10/site-packages/pandas/io/parquet.py:190: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


In [5]:
technology = 'Xenium'

# Construct data directory
data_dir = os.path.join(data_root_dir, sample)

# Make cell image coordinates
path_transformation_matrix = os.path.join(path_landscape_files, 'micron_to_image_transform.csv')
path_meta_cell_micron = os.path.join(data_dir, 'cells.csv.gz')
path_meta_cell_image = os.path.join(path_landscape_files, 'cell_metadata.parquet')

# Generate transcript tiles
print("\n========Generating transcript tiles========")
path_trx = os.path.join(data_dir, 'transcripts.parquet')
path_trx_tiles = os.path.join(path_landscape_files, 'transcript_tiles')


# tile_bounds = dega.pre.make_trx_tiles(
#     technology,
#     path_trx,
#     path_transformation_matrix,
#     path_trx_tiles,
#     coarse_tile_factor=10,
#     tile_size=tile_size,
#     chunk_size=100000,
#     verbose=False,
#     image_scale=1,
#     max_workers=2
# )
# print (f"tile bounds: {tile_bounds}")

# import pandas as pd
# df = pd.read_parquet(f"{path_trx_tiles}/transcripts_tile_0_49.parquet")
# df.head()



========Generating transcript tiles========


In [6]:
tile_bounds = {'x_min': 0, 'x_max': 34126.65, 'y_min': 0, 'y_max': 13744.4}


# Generate boundary tiles
print("\n========Generating boundary tiles========")
path_cell_boundaries = os.path.join(data_dir, 'cell_boundaries.parquet')
path_output = os.path.join(path_landscape_files, 'cell_segmentation')
cells_orig = dega.pre.make_cell_boundary_tiles(
    technology,
    path_cell_boundaries,
    path_meta_cell_micron,
    path_transformation_matrix,
    path_output,
    coarse_tile_factor=10,
    tile_size=tile_size,
    tile_bounds=tile_bounds,
    image_scale=1,
    max_workers=2
)

import pandas as pd
df = pd.read_parquet(f"{path_output}/cell_tile_0_49.parquet")
df.head()


========Generating boundary tiles========


Processing coarse tiles: 100%|██████████████████| 14/14 [00:34<00:00,  2.50s/it]


,GEOMETRY,name
0,"[[[247.0, 12469.0], [246.0, 12470.0], [241.0, ...",49751
1,"[[[210.0, 12463.0], [205.0, 12464.0], [201.0, ...",49753
2,"[[[125.0, 12472.0], [123.0, 12474.0], [119.0, ...",49758
3,"[[[172.0, 12466.0], [168.0, 12470.0], [166.0, ...",49759
4,"[[[146.0, 12447.0], [143.0, 12449.0], [135.0, ...",49761


In [ ]:
"""
Pancreas

boundary_tile
- Before: 1m 8s
- After mapping name with integer using  pre-generated dictionary during writing parquet file: 9m 15s
- After mapping name with integer using  pre-generated dictionary after writing parquet file: 1m 20s + 1m 8s (original) = 2m 28s

trx_tile
- Before: 1m
- After mapping name with integer using  pre-generated dictionary during writing parquet file:  <- update
- After mapping name with integer using  pre-generated dictionary after writing parquet file: 3m 52s + 1m (original) = 4m 52s


Ovarian cancer on GCP

After mapping name with integer using  pre-generated dictionary after writing parquet file:
cell_tile: 28min
trx_tile: 21min
"""

In [39]:
# # sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

# def get_directory_size_in_mb(directory):
#     total_size_bytes = 0
#     for dirpath, dirnames, filenames in os.walk(directory):
#         for filename in filenames:
#             filepath = os.path.join(dirpath, filename)
#             total_size_bytes += os.path.getsize(filepath)
#     # Convert bytes to megabytes (1 MB = 1024 * 1024 bytes)
#     total_size_mb = total_size_bytes / (1024 * 1024)
#     return total_size_mb

# # Example usage
# directory_path = f'landscape_files/{sample}/transcript_tiles'
# size_in_mb = get_directory_size_in_mb(directory_path)
# print(f"Total size of '{directory_path}': {size_in_mb:.2f} MB")


## Visualize Landscape files in Celldega

In [24]:
landscape_data_dir = 'landscape_files'
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'

server_address = dega.viz.get_local_server()

landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{server_address}/{landscape_data_dir}/{sample}",
)

landscape_ist

Server running on port 53272


Landscape(base_url='http://localhost:53272/landscape_files/Xenium_V1_human_Pancreas_FFPE_outs', technology='Xe…

In [31]:
import pandas as pd
df = pd.read_parquet(f"{path_landscape_files}/cell_segmentation/cell_tile_0_49.parquet")
df.head()

,GEOMETRY,name
0,"[[[247.0, 12469.0], [246.0, 12470.0], [241.0, ...",49751
1,"[[[210.0, 12463.0], [205.0, 12464.0], [201.0, ...",49753
2,"[[[125.0, 12472.0], [123.0, 12474.0], [119.0, ...",49758
3,"[[[172.0, 12466.0], [168.0, 12470.0], [166.0, ...",49759
4,"[[[146.0, 12447.0], [143.0, 12449.0], [135.0, ...",49761


In [16]:
import pandas as pd
df_meta_cell = pd.read_parquet(f"{path_landscape_files}/cell_metadata.parquet")
df_meta_cell.tail()

,name,geometry
140697,oiloppgp-1,"[28624.35447082031, 2612.436901953247]"
140698,oilpccne-1,"[28738.348028447264, 2329.1849700721436]"
140699,oimacfoj-1,"[28616.427081708985, 2949.374491572632]"
140700,oimaiaae-1,"[28379.26717302539, 2524.7217775576173]"
140701,oimajkkk-1,"[28341.820025098634, 2700.1612175246582]"
